In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
CHECKPOINT_PATH = os.path.join(SPARKYLLM, "checkpoints", "gpt_medium_phase2.pth")
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")
META_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "train_long_meta.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ==================== MODEL DEFINITION ====================
BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64
FF_HIDDEN_DIM = 4 * EMBED_DIM

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList([TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.0) for _ in range(NUM_LAYERS)])
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks: x = block(x)
        return self.lm_head(self.final_norm(x))

# ==================== LOAD MODEL ====================
with open(META_PATH, "r") as f:
    meta = json.load(f)
VOCAB_SIZE = int(meta["vocab_size"])
if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleGPT(VOCAB_SIZE).to(device)

# Load checkpoint with diagnostics
if not os.path.exists(CHECKPOINT_PATH):
    print(f"CHECKPOINT NOT FOUND: {CHECKPOINT_PATH}")
    print("Listing available checkpoints:")
    ckpt_dir = os.path.dirname(CHECKPOINT_PATH)
    if os.path.isdir(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            print(f"  {f}")
    else:
        print(f"  Directory does not exist: {ckpt_dir}")
else:
    file_size = os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024
    print(f"Checkpoint: {CHECKPOINT_PATH} ({file_size:.1f} MB)")

    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    print(f"Checkpoint keys: {len(state_dict)}")

    # Show a few raw keys to understand the format
    raw_keys = list(state_dict.keys())[:5]
    print(f"Sample keys: {raw_keys}")

    # Clean _orig_mod. prefix from torch.compile
    clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

    # Match against model
    model_state = model.state_dict()
    matched = {}
    mismatched_shape = []
    missing_in_ckpt = []
    extra_in_ckpt = []

    for k, v in clean_state.items():
        if k in model_state:
            if v.shape == model_state[k].shape:
                matched[k] = v
            else:
                mismatched_shape.append((k, v.shape, model_state[k].shape))
        else:
            extra_in_ckpt.append(k)

    for k in model_state:
        if k not in clean_state:
            missing_in_ckpt.append(k)

    print(f"\nMatched:          {len(matched)} / {len(model_state)} model keys")
    if mismatched_shape:
        print(f"Shape mismatch:   {len(mismatched_shape)}")
        for k, ckpt_shape, model_shape in mismatched_shape:
            print(f"  {k}: checkpoint {ckpt_shape} vs model {model_shape}")
    if missing_in_ckpt:
        print(f"Missing in ckpt:  {len(missing_in_ckpt)}")
        for k in missing_in_ckpt[:10]:
            print(f"  {k}")
    if extra_in_ckpt:
        print(f"Extra in ckpt:    {len(extra_in_ckpt)}")
        for k in extra_in_ckpt[:10]:
            print(f"  {k}")

    if len(matched) == 0:
        print("\nERROR: No keys matched! The checkpoint is incompatible with this model.")
    else:
        model.load_state_dict(matched, strict=False)
        print(f"\nLoaded {len(matched)} layers successfully.")

model.eval()

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
eot_id = tokenizer.token_to_id("<|endoftext|>")

print(f"\nModel on {device} | Vocab: {VOCAB_SIZE} | Params: {sum(p.numel() for p in model.parameters()):,}")

In [5]:
@torch.no_grad()
def generate(prompt, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9):
    ids = tokenizer.encode(prompt).ids
    tokens = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_tokens):
        context = tokens[:, -BLOCK_SIZE:]
        logits = model(context)[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Top-k
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # Top-p (nucleus)
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = float('-inf')
            logits = torch.zeros_like(logits).scatter(1, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        if next_token.item() == eot_id:
            break

        tokens = torch.cat([tokens, next_token], dim=1)

    output_ids = tokens[0].tolist()
    return tokenizer.decode(output_ids)

In [6]:
# ==================== TEST 1: PERPLEXITY ====================
# Measures how well the model predicts held-out text. Lower = better.
# Rough guide: <50 decent, <20 good, <10 very good for small LMs.
import numpy as np
import math

EVAL_FILE = os.path.join(SPARKYLLM, "datasets_pretrain", "training_data_long.txt")
EVAL_CHUNKS = 50
CHUNK_LENGTH = BLOCK_SIZE

@torch.no_grad()
def compute_perplexity(text_path, num_chunks, chunk_len):
    # Stream-tokenize line by line to avoid loading everything into memory
    all_ids = []
    target_tokens = (num_chunks + 1) * (chunk_len + 1)

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ids = tokenizer.encode(line).ids
            all_ids.extend(ids)
            if len(all_ids) >= target_tokens:
                break

    total_tokens = len(all_ids)
    print(f"Tokenized {total_tokens:,} tokens for evaluation")

    if total_tokens < chunk_len + 1:
        print("Not enough tokens to evaluate!")
        return None

    losses = []
    rng = np.random.default_rng(42)
    starts = rng.integers(0, total_tokens - chunk_len - 1, size=num_chunks)

    for s in starts:
        chunk = all_ids[s : s + chunk_len + 1]
        x = torch.tensor([chunk[:-1]], dtype=torch.long, device=device)
        y = torch.tensor([chunk[1:]], dtype=torch.long, device=device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())

    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss)

    print(f"\nPerplexity Results ({num_chunks} chunks of {chunk_len} tokens):")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Perplexity:   {ppl:.2f}")
    print(f"  Min Loss:     {min(losses):.4f} (best chunk)")
    print(f"  Max Loss:     {max(losses):.4f} (worst chunk)")
    return ppl

perplexity = compute_perplexity(EVAL_FILE, EVAL_CHUNKS, CHUNK_LENGTH)

Tokenized 39,328 tokens for evaluation

Perplexity Results (50 chunks of 768 tokens):
  Average Loss: 9.1729
  Perplexity:   9632.69
  Min Loss:     8.9373 (best chunk)
  Max Loss:     9.4300 (worst chunk)


In [7]:
# ==================== TEST 2: REPETITION RATE ====================
# Measures how often the model repeats itself. Lower = better.
# Checks 2-gram, 3-gram, and 4-gram repetition rates.

from collections import Counter

def repetition_rate(text, n):
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(ngrams) if ngrams else 0.0

test_prompts = [
    "Once upon a time",
    "The king looked at the horizon and",
    "In the beginning there was",
    "She walked through the forest",
    "The old man sat by the fire and",
]

print("Repetition Rate Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'2-gram':>8} {'3-gram':>8} {'4-gram':>8}")
print("-" * 70)

all_outputs = []
for prompt in test_prompts:
    output = generate(prompt, max_tokens=300, temperature=0.8, top_k=40)
    all_outputs.append(output)
    r2 = repetition_rate(output, 2)
    r3 = repetition_rate(output, 3)
    r4 = repetition_rate(output, 4)
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {r2:>7.1%} {r3:>7.1%} {r4:>7.1%}")

# Overall
combined = " ".join(all_outputs)
print("-" * 70)
print(f"{'OVERALL':<40} {repetition_rate(combined, 2):>7.1%} {repetition_rate(combined, 3):>7.1%} {repetition_rate(combined, 4):>7.1%}")
print("\nGuide: <10% is good, 10-25% is okay, >25% is concerning")

Repetition Rate Analysis
Prompt                                     2-gram   3-gram   4-gram
----------------------------------------------------------------------
Once upon a time                            0.0%    0.0%    0.0%
The king looked at the horizon and          0.0%    0.0%    0.0%
In the beginning there was                  1.1%    0.0%    0.0%
She walked through the forest               0.6%    0.0%    0.0%
The old man sat by the fire and             0.5%    0.0%    0.0%
----------------------------------------------------------------------
OVERALL                                     0.7%    0.0%    0.0%

Guide: <10% is good, 10-25% is okay, >25% is concerning


In [8]:
# ==================== TEST 3: TOKEN DIVERSITY ====================
# Measures unique tokens / total tokens. Higher = better.
# Low diversity suggests mode collapse (model stuck in a rut).

def token_diversity(text):
    ids = tokenizer.encode(text).ids
    total = len(ids)
    unique = len(set(ids))
    return unique, total, unique / total if total > 0 else 0.0

print("Token Diversity Analysis")
print("=" * 70)
print(f"{'Prompt':<40} {'Unique':>7} {'Total':>7} {'Ratio':>8}")
print("-" * 70)

total_unique_all = set()
total_count_all = 0

for prompt, output in zip(test_prompts, all_outputs):
    unique, total, ratio = token_diversity(output)
    ids = tokenizer.encode(output).ids
    total_unique_all.update(ids)
    total_count_all += total
    label = prompt[:37] + "..." if len(prompt) > 37 else prompt
    print(f"{label:<40} {unique:>7} {total:>7} {ratio:>7.1%}")

print("-" * 70)
overall_ratio = len(total_unique_all) / total_count_all if total_count_all > 0 else 0
print(f"{'OVERALL':<40} {len(total_unique_all):>7} {total_count_all:>7} {overall_ratio:>7.1%}")
print(f"\nVocab utilization: {len(total_unique_all)} / {VOCAB_SIZE} tokens used ({len(total_unique_all)/VOCAB_SIZE:.1%})")
print("Guide: >30% diversity is good, <15% is concerning")

Token Diversity Analysis
Prompt                                    Unique   Total    Ratio
----------------------------------------------------------------------
Once upon a time                             200     304   65.8%
The king looked at the horizon and           145     309   46.9%
In the beginning there was                   121     303   39.9%
She walked through the forest                167     311   53.7%
The old man sat by the fire and              158     305   51.8%
----------------------------------------------------------------------
OVERALL                                      704    1532   46.0%

Vocab utilization: 704 / 15040 tokens used (4.7%)
Guide: >30% diversity is good, <15% is concerning


In [9]:
# ==================== TEST 4: COHERENCE CHECK ====================
# Generates multiple continuations from the same prompt.
# If the model is coherent, outputs should be varied but thematically consistent.

coherence_prompts = [
    "The knight drew his sword and",
    "Deep in the mountains there lived",
]

NUM_SAMPLES = 3

for prompt in coherence_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    for i in range(NUM_SAMPLES):
        output = generate(prompt, max_tokens=150, temperature=0.9, top_k=50)
        # Remove the prompt portion for cleaner display
        continuation = output[len(tokenizer.decode(tokenizer.encode(prompt).ids)):]
        print(f"\n--- Sample {i+1} ---")
        print(continuation.strip()[:500])


PROMPT: The knight drew his sword and

--- Sample 1 ---
HadHad Piet fifty deleg setting Châteaurons minds producingited fant dus deleg resc int San press educationιjolras� transactionface harm monasteries hayaras Mark fant fifty Retrav ear vibration commission tribun harm GodefroyUL abhorgian miracle bewildered dus Sept hellf ruff enem Val miracle press Probus authent eating Smith Smith enterprise trou merch prophecy conspiracy gilt obvious Being excluded wrest dest jobigrête worked managerland worked sunsh Office fuelouis cons medical resc madness w

--- Sample 2 ---
paintisp claimed gent successfully precariousESS press quietlyishop smellESS esc prudrys prohibitedchester�nine harm coffee attempt and� quietly the enem eloquenceacent press Queen Queen pleasuresmen presschester magaz sangu rushoura sure brid date sont vain Villefort smellchester harpoone serpent clergy Cod press precarious urban, abilitiesavo Piet Pantructed bought dungeon excluded attempteges precarious fiftyavouchves

In [10]:
# ==================== TEST 5: LOSS ON KNOWN vs RANDOM TEXT ====================
# Compares model loss on in-domain text (from training data) vs out-of-domain text.
# The model should have LOWER loss on in-domain text.
# If losses are similar, the model hasn't learned your data well.

import random

@torch.no_grad()
def compute_loss_on_text(text):
    ids = tokenizer.encode(text).ids
    if len(ids) < 2:
        return None
    ids = ids[:BLOCK_SIZE + 1]
    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([ids[1:]], dtype=torch.long, device=device)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
    return loss.item()

# Load passages from training data (stream to avoid OOM)
rng = random.Random(42)
in_domain_passages = []
with open(EVAL_FILE, "r", encoding="utf-8", errors="ignore") as f:
    full_text = f.read(500_000)  # read first 500KB only

for _ in range(5):
    start = rng.randint(0, max(0, len(full_text) - 5000))
    passage = full_text[start:start + 3000].strip()
    if passage:
        in_domain_passages.append(passage)

# Out-of-domain: longer passages across different domains the model has never seen
out_of_domain_passages = [
    "The quantum computational framework leverages topological qubits to achieve "
    "fault-tolerant error correction in superconducting circuits operating at millikelvin "
    "temperatures. Recent advances in surface code architectures have demonstrated that "
    "logical error rates can be suppressed exponentially by increasing the code distance, "
    "provided physical error rates remain below the threshold. The integration of classical "
    "decoding algorithms with real-time feedback loops presents significant engineering "
    "challenges that must be addressed before practical quantum advantage can be realized "
    "in applications such as molecular simulation and cryptographic analysis.",

    "SELECT u.name, u.email, COUNT(o.id) as order_count, SUM(o.total) as lifetime_value "
    "FROM users u INNER JOIN orders o ON u.id = o.user_id LEFT JOIN refunds r ON o.id = r.order_id "
    "WHERE o.created_at BETWEEN '2024-01-01' AND '2024-12-31' AND r.id IS NULL "
    "GROUP BY u.name, u.email HAVING SUM(o.total) > 1000 ORDER BY lifetime_value DESC LIMIT 50; "
    "CREATE INDEX idx_orders_user_date ON orders(user_id, created_at) INCLUDE (total); "
    "EXPLAIN ANALYZE SELECT * FROM orders WHERE user_id = 42 AND status = 'completed';",

    "La transformada de Fourier discreta permite descomponer una señal temporal en sus "
    "componentes frecuenciales, facilitando el análisis espectral en procesamiento digital "
    "de señales. El teorema de muestreo de Nyquist-Shannon establece que una señal continua "
    "puede ser reconstruida perfectamente a partir de sus muestras discretas si la frecuencia "
    "de muestreo es al menos el doble de la frecuencia máxima presente en la señal. Las "
    "aplicaciones prácticas incluyen compresión de audio, procesamiento de imágenes médicas "
    "y sistemas de comunicación inalámbrica de quinta generación.",

    "def fibonacci(n, memo={}): "
    "    if n in memo: return memo[n]\n"
    "    if n <= 1: return n\n"
    "    memo[n] = fibonacci(n-1, memo) + fibonacci(n-2, memo)\n"
    "    return memo[n]\n\n"
    "class BinarySearchTree:\n"
    "    def __init__(self, value):\n"
    "        self.value = value\n"
    "        self.left = None\n"
    "        self.right = None\n"
    "    def insert(self, val):\n"
    "        if val < self.value:\n"
    "            if self.left is None: self.left = BinarySearchTree(val)\n"
    "            else: self.left.insert(val)\n"
    "        else:\n"
    "            if self.right is None: self.right = BinarySearchTree(val)\n"
    "            else: self.right.insert(val)\n",

    "The 2024 fiscal year report indicates a 12.3% increase in EBITDA margins driven "
    "primarily by operational efficiencies in the supply chain management division across "
    "APAC regions. Gross merchandise volume reached $4.2 billion, representing a 23% "
    "year-over-year increase despite macroeconomic headwinds including rising interest rates "
    "and currency depreciation in emerging markets. The board has approved a $500 million "
    "share repurchase program and increased the quarterly dividend to $0.85 per share, "
    "reflecting confidence in sustained free cash flow generation through fiscal 2025.",
]

print("Loss Comparison: In-Domain vs Out-of-Domain")
print("=" * 60)

print("\nIn-Domain (training data passages):")
in_losses = []
for i, passage in enumerate(in_domain_passages):
    loss = compute_loss_on_text(passage)
    if loss is not None:
        in_losses.append(loss)
        preview = passage[:60].replace('\n', ' ') + "..."
        print(f"  [{i+1}] Loss: {loss:.4f}  |  {preview}")

print("\nOut-of-Domain (unseen text):")
out_losses = []
domains = ["Quantum Computing", "SQL", "Spanish/DSP", "Python Code", "Finance"]
for i, (passage, domain) in enumerate(zip(out_of_domain_passages, domains)):
    loss = compute_loss_on_text(passage)
    if loss is not None:
        out_losses.append(loss)
        print(f"  [{i+1}] Loss: {loss:.4f}  |  [{domain}]")

avg_in = sum(in_losses) / len(in_losses) if in_losses else 0
avg_out = sum(out_losses) / len(out_losses) if out_losses else 0
gap = avg_out - avg_in

print(f"\n{'='*60}")
print(f"  Avg In-Domain Loss:      {avg_in:.4f}  (PPL: {math.exp(avg_in):.2f})")
print(f"  Avg Out-of-Domain Loss:  {avg_out:.4f}  (PPL: {math.exp(avg_out):.2f})")
print(f"  Gap:                     {gap:.4f}")
print(f"\nVerdict: ", end="")
if gap > 1.0:
    print("GOOD - model clearly learned in-domain patterns")
elif gap > 0.3:
    print("OK - model shows some domain specialization")
elif gap > 0:
    print("WEAK - model barely distinguishes in-domain from random text")
else:
    print("BAD - model performs worse on its own training data")

Loss Comparison: In-Domain vs Out-of-Domain

In-Domain (training data passages):
  [1] Loss: 9.3886  |  d their host.  “Is there anyone we have not met? My wife and...
  [2] Loss: 9.4960  |  his own part in it.  To him life was more than meat and the ...
  [3] Loss: 9.1558  |  s true of the manasic globe, and of our manasa.  The great l...
  [4] Loss: 9.0913  |  elightful. The voice of a man muffled up and covered with sn...
  [5] Loss: 9.5332  |  ho approach us with offers to donate.  International donatio...

Out-of-Domain (unseen text):
  [1] Loss: 9.7057  |  [Quantum Computing]
  [2] Loss: 9.7160  |  [SQL]
  [3] Loss: 9.7797  |  [Spanish/DSP]
  [4] Loss: 9.6544  |  [Python Code]
  [5] Loss: 9.6226  |  [Finance]

  Avg In-Domain Loss:      9.3330  (PPL: 11304.70)
  Avg Out-of-Domain Loss:  9.6957  (PPL: 16247.55)
  Gap:                     0.3627

Verdict: OK - model shows some domain specialization


In [ ]:
# ==================== INTERACTIVE CHAT ====================
# Type text, the model continues it. Context accumulates across turns.
# Type "reset" to clear context, "quit" to stop.

@torch.no_grad()
def generate_streaming(token_ids, max_tokens=150, temperature=0.8, top_k=40):
    """Generate from a list of token IDs, print as it goes, return updated list."""
    tokens = torch.tensor([token_ids], dtype=torch.long, device=device)
    new_ids = []

    for _ in range(max_tokens):
        context = tokens[:, -BLOCK_SIZE:]
        logits = model(context)[:, -1, :]
        logits = logits / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tid = next_token.item()

        if tid == eot_id:
            print("<|end|>", end="")
            break

        new_ids.append(tid)
        tokens = torch.cat([tokens, next_token], dim=1)
        piece = tokenizer.decode([tid])
        print(piece, end="", flush=True)

    print()
    return token_ids + new_ids

# --- Main loop ---
context_ids = []
print("=" * 60)
print("INTERACTIVE MODE")
print("  Your text gets added to context, model continues from there.")
print("  Commands: 'reset' = clear context, 'quit' = exit")
print("=" * 60)

while True:
    try:
        user_input = input("\nYou > ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nExiting.")
        break

    if user_input.lower() == "quit":
        break
    if user_input.lower() == "reset":
        context_ids = []
        print("Context cleared.")
        continue
    if not user_input:
        continue

    # Tokenize user input and append to context
    new_tokens = tokenizer.encode(user_input).ids
    context_ids.extend(new_tokens)

    # Trim context if too long (keep last BLOCK_SIZE tokens)
    if len(context_ids) > BLOCK_SIZE:
        context_ids = context_ids[-BLOCK_SIZE:]

    print(f"\nModel ({len(context_ids)} tokens in context) > ", end="", flush=True)
    context_ids = generate_streaming(context_ids, max_tokens=150, temperature=0.8, top_k=40)
    print(f"  [{len(context_ids)} tokens total]")